# HGP-clusterer : 3D puis 4D Panoptic Segmentation sur SemanticKITTI

Ce notebook implémente un pipeline de segmentation panoptique en "streaming" en utilisant **HGP-clusterer**.

**Pipeline :**
1.  **Setup** : Installation des dépendances.
2.  **Data** : Chargement d'une séquence SemanticKITTI.
3.  **Preprocessing** : Construction des nuages de points.
4.  **Clustering & Tracking (Streaming)** : 
    - Initialisation (Frame 0) : Segmentation d'instance 3D (ou BEV) et création des représentations des clusters (volume, classe, vitesse).
    - Suivi (Frames suivantes) : *En cours (itération frame par frame)*.
5.  **Evaluation** : *Adaptée pour le suivi*.
6.  **Visualisation** : Rendu interactif.

In [ ]:
# @title 1.1 Choix du Backend Géométrique
# 'geogram' est recommandé pour la vitesse (headless). 'cgal' est plus lent mais exact.
BACKEND = 'cgal'  # @param ['geogram', 'cgal']
print(f"Backend sélectionné : {BACKEND}")

In [ ]:
# @title 1.2 Installation des dépendances système
!apt-get update -qq
!apt-get install -y -qq build-essential cmake git libeigen3-dev libomp-dev

if BACKEND == 'cgal':
    # libboost-all-dev est souvent nécessaire pour que CMake détecte correctement CGAL
    !apt-get install -y -qq libcgal-dev libtbb-dev libtbbmalloc2 libgmp-dev libmpfr-dev libboost-all-dev

In [ ]:
# @title 1.3 Installation des dépendances Python
!pip install -q --upgrade pip setuptools wheel Cython cmake jedi gdown pybind11
!pip install -q numpy scipy scikit-learn plotly tqdm joblib open3d plyfile hdbscan pandas matplotlib pyyaml shapely

In [ ]:
%%bash
# @title 1.4 Installation de HGP-clusterer et SemanticKITTI-API
set -euo pipefail
WORKDIR="/content"
mkdir -p "${WORKDIR}"
cd "${WORKDIR}"

# HGP-clusterer
if [ -d HGP-clusterer ]; then
    git -C HGP-clusterer pull --ff-only
else
    git clone https://github.com/Ludwig-H/HGP-clusterer.git
fi

# SemanticKITTI API (pour l'évaluation)
if [ -d semantic-kitti-api ]; then
    git -C semantic-kitti-api pull --ff-only
else
    git clone https://github.com/PRBonn/semantic-kitti-api.git
fi

In [ ]:
# @title 1.5 Compilation de HGP
import os
import sys
import subprocess

WORKDIR = "/content"
os.chdir(WORKDIR)

if BACKEND == 'geogram':
    if not os.path.exists('geogram'):
        print("Clonage de Geogram...")
        !git clone --recursive https://github.com/BrunoLevy/geogram.git
    
    print("Compilation de Geogram (Headless)...")
    !cmake -S geogram -B geogram/build -DCMAKE_BUILD_TYPE=Release -DGEOGRAM_WITH_GRAPHICS=OFF -DGEOGRAM_WITH_LUA=OFF -DGEOGRAM_WITH_GARGANTUA=OFF
    !cmake --build geogram/build --config Release --parallel 4
    !cmake --install geogram/build --prefix /usr/local
    os.environ['GEOGRAM_INSTALL_PREFIX'] = '/usr/local'

elif BACKEND == 'cgal':
    print("Configuration CGAL...")
    
    # -- FIX: Add CGAL path to environment for setup_cgal.py --
    cgal_prefix = "/usr/lib/x86_64-linux-gnu/cmake/CGAL"
    current_cpp = os.environ.get("CMAKE_PREFIX_PATH", "")
    os.environ["CMAKE_PREFIX_PATH"] = f"{current_cpp}:{cgal_prefix}" if current_cpp else cgal_prefix
    # ---------------------------------------------------------

    # On tente de construire l'outil CGAL, mais on continue même en cas d'erreur
    # car le setup.py principal pourrait réussir autrement.
    try:
        subprocess.run(["python3", f"{WORKDIR}/HGP-clusterer/scripts/setup_cgal.py"], check=True)
    except subprocess.CalledProcessError:
        print("⚠️ Attention: Echec du script setup_cgal.py. Tentative de continuation avec le build principal...")

os.chdir(f"{WORKDIR}/HGP-clusterer")
!rm -rf build dist *.egg-info

install_cmd = "pip install --no-build-isolation -v --no-deps ."
if BACKEND == 'geogram':
    install_cmd = f"GEOGRAM_INSTALL_PREFIX=/usr/local {install_cmd}"
elif BACKEND == 'cgal':
    # Ajout du chemin système pour CGAL (Debian/Ubuntu/Colab)
    # Note: On le passe aussi explicitement ici pour être sûr
    install_cmd = f"CGALDELAUNAY_ROOT={WORKDIR}/HGP-clusterer/CGALDelaunay CMAKE_PREFIX_PATH={WORKDIR}/HGP-clusterer:{cgal_prefix} {install_cmd}"

print(f"Exécution : {install_cmd}")
!{install_cmd}

os.environ["CGALDELAUNAY_ROOT"] = f"{WORKDIR}/HGP-clusterer/CGALDelaunay"

try:
    from hgp_clusterer import HGPClusterer
    print("✅ HGPClusterer installé.")
except ImportError as e:
    print(f"❌ Erreur import HGP: {e}")

In [ ]:
# @title 2.1 Configuration Séquence et Téléchargement
# IMPORTANT : Si vous ne voulez tester qu'une seule séquence, lancez cette cellule.
# Le téléchargement via gdown --folder récupère tout le dossier si on ne filtre pas.
# Ici, on télécharge tout le dataset SemanticKITTI (partiel) fourni via le lien Drive.

SEQUENCE_TO_TEST = 8 # @param {type:"integer"}
DOWNLOAD_DATA = True # @param {type:"boolean"}

START_FRAME = 0 # @param {type:"integer"}
NUM_FRAMES = 10 # @param {type:"integer"} (-1 pour toutes les frames)
DT_SCALE = 0.5  # @param {type:"number"}
APPLY_BEV = False # @param {type:"boolean"}
# Mode Sémantique :
# - 'Oracle' : Utilise la vérité terrain fournie par SemanticKITTI pour filtrer les objets mobiles (Things).
# - 'None' : Ne filtre rien (Lance le clustering sur absolument toute la scène, lent et non recommandé).
# (Note: Le chargement d'une prédiction réseau externe viendrait ici dans une future mise à jour)
SEMANTIC_MODE = "Oracle" # @param ["Oracle", "None"]

# Choix du mode de téléchargement :
# - 'Folder' : Télécharge fichier par fichier (Très lent pour 10k fichiers, mais utile si on a que le lien du dossier)
# - 'Zip' : Télécharge une archive unique et décompresse (Beaucoup plus rapide, recommandé)
DOWNLOAD_MODE = "Folder" # @param ["Folder", "Zip"]

# IDs Google Drive par séquence
# Remplissez ce dictionnaire avec les IDs des dossiers ou des zips pour chaque séquence.
SEQUENCE_DRIVE_IDS = {
    8: {
        "Folder": "1UqFKvekjyic6L_8KD1kcv8MuGmQMIk0A",
        "Zip": "1ZoZtzdFAkPWYHT8sFsHjwyEmFHbpaIQH"
    }
}

# Dossier Racine (Fallback si ID spécifique non trouvé en mode Folder)
ROOT_FOLDER_ID = "1ORVzSo-TWbNHeAC0-k3mxX9AiJHI_tVu"

if DOWNLOAD_DATA:
    import os
    import shutil
    
    # Destination racine
    base_dest = "/content/semantic_kitti_data"
    seq_str = f"{SEQUENCE_TO_TEST:02d}"
    target_dir = os.path.join(base_dest, seq_str)
    
    if not os.path.exists(target_dir):
        print(f"Démarrage du téléchargement (Mode : {DOWNLOAD_MODE})...")
        
        # Récupération des IDs pour la séquence choisie
        seq_ids = SEQUENCE_DRIVE_IDS.get(SEQUENCE_TO_TEST, {})
        
        if DOWNLOAD_MODE == "Zip":
            zip_id = seq_ids.get("Zip")
            if not zip_id:
                print(f"⚠️ Aucun ID Zip trouvé pour la séquence {SEQUENCE_TO_TEST}. Veuillez remplir SEQUENCE_DRIVE_IDS.")
                print("Passage automatique en mode Folder (Fallback)...")
                # On ne lance pas d'erreur, on essaie le folder si possible, sinon root
                DOWNLOAD_MODE = "Folder" 
            else:
                print(f"Téléchargement de l'archive Zip (ID: {zip_id})...")
                zip_path = os.path.join(base_dest, "sequence.zip")
                os.makedirs(base_dest, exist_ok=True)
                
                # Téléchargement
                !gdown {zip_id} -O {zip_path} --quiet
                
                print("Décompression...")
                # On décompresse
                !unzip -q {zip_path} -d {target_dir}
                !rm {zip_path}
                
                # Vérification de la structure (si le zip contenait un sous-dossier, on remonte)
                if not os.path.exists(os.path.join(target_dir, "velodyne")):
                    # Tentative de correction automatique
                    sub_dirs = [d for d in os.listdir(target_dir) if os.path.isdir(os.path.join(target_dir, d))]
                    if len(sub_dirs) == 1:
                        inner_dir = os.path.join(target_dir, sub_dirs[0])
                        print(f"Structure imbriquée détectée, déplacement de {inner_dir} vers {target_dir}...")
                        for item in os.listdir(inner_dir):
                            shutil.move(os.path.join(inner_dir, item), target_dir)
                        os.rmdir(inner_dir)

        # Note: Ce bloc est exécuté si mode Folder OU si fallback depuis Zip
        if DOWNLOAD_MODE == "Folder":
            folder_id = seq_ids.get("Folder")
            if not folder_id:
                # Fallback sur le root folder (pas idéal mais fonctionnel)
                print(f"ID spécifique Folder manquant pour la séquence {SEQUENCE_TO_TEST}.")
                folder_id = ROOT_FOLDER_ID
                dl_target = base_dest
            else:
                dl_target = target_dir

            print(f"Téléchargement du dossier (ID: {folder_id})...")
            # --- TÉLÉCHARGEMENT SÉLECTIF ---
            try:
                import gdown
                print("Analyse du contenu du Drive... (limité aux 50 premiers fichiers par dossier par Google Drive)")
                # remaining_ok=True permet de récupérer les 50 premiers fichiers sans crasher
                files = gdown.download_folder(id=folder_id, skip_download=True, quiet=True, remaining_ok=True)
                if files:
                    needed_frames = set()
                    if NUM_FRAMES != -1:
                        needed_frames = set(range(START_FRAME, START_FRAME + NUM_FRAMES))
                    
                    downloaded_count = 0
                    
                    for f in files:
                        try:
                            f_path = f.path if hasattr(f, 'path') else f.get('path', '')
                            f_id = f.id if hasattr(f, 'id') else f.get('id', '')
                        except:
                            continue
                            
                        # Vérifier si c'est un fichier lié à une frame
                        filename = f_path.split('/')[-1] if '/' in f_path else f_path
                        is_frame_file = False
                        frame_idx = -1
                        
                        if filename.endswith('.bin') or filename.endswith('.label'):
                            try:
                                frame_idx = int(filename.split('.')[0])
                                is_frame_file = True
                            except ValueError:
                                pass
                                
                        # Filtrer
                        if is_frame_file and NUM_FRAMES != -1:
                            if frame_idx not in needed_frames:
                                continue # On skip cette frame
                        
                        # Créer le chemin local
                        import os
                        local_path = os.path.join(dl_target, f_path)
                        os.makedirs(os.path.dirname(local_path), exist_ok=True)
                        
                        if not os.path.exists(local_path):
                            print(f"Téléchargement : {f_path}")
                            gdown.download(id=f_id, output=local_path, quiet=True)
                        downloaded_count += 1
                        
                    print(f"Téléchargement sélectif terminé ({downloaded_count} fichiers traités).")
                    
                    # Vérification si on a pu tout récupérer ou si on a tapé la limite des 50
                    # On s'attend à 2 fichiers par frame (bin + label) s'il s'agit de frames
                    # Mais s'il y a des fichiers autres (calib), on ne s'inquiète que si downloaded_count est faible.
                    # Pour être simple, on informe l'utilisateur si la frame demandée max n'est pas dans les 50.
                    if NUM_FRAMES != -1 and (START_FRAME + NUM_FRAMES) > 50:
                        print(f"\n⚠️ ATTENTION : Vous avez demandé jusqu'à la frame {START_FRAME + NUM_FRAMES - 1}.")
                        print("Google Drive limite le listage anonyme aux 50 premiers fichiers de chaque dossier (jusqu'à la frame 49).")
                        print("👉 Pour traiter au-delà de la frame 49, VEUILLEZ UTILISER LE MODE 'Zip' dans les paramètres.\n")
                else:
                    print("Impossible d'analyser le dossier. Veuillez utiliser le mode Zip.")
            except Exception as e:
                print(f"\n❌ Le téléchargement sélectif a échoué ({e}).")
                print("👉 VEUILLEZ UTILISER LE MODE 'Zip' DANS LES PARAMÈTRES CI-DESSUS.\n")
        print("Téléchargement terminé.")
    else:
        print(f"Dossier {target_dir} existe déjà. Skip download.")
else:
    print("Téléchargement désactivé.")

print(f"Séquence cible pour le test : {SEQUENCE_TO_TEST}")

In [ ]:
# @title 2.2 Loader SemanticKITTI
import os
import numpy as np
import glob

class SemanticKITTILoader:
    def __init__(self, base_path, sequence_num):
        self.seq_str = f"{sequence_num:02d}"

        # Recherche du dossier de la séquence.
        # Structure attendue : base_path/08 ou base_path/sequences/08

        # 1. Chercher direct
        possible_paths = glob.glob(f"{base_path}/{self.seq_str}")

        # 2. Chercher dans un sous-dossier 'sequences' (structure officielle KITTI)
        if not possible_paths:
            possible_paths = glob.glob(f"{base_path}/**/sequences/{self.seq_str}", recursive=True)

        # 3. Chercher récursivement n'importe où (au cas où gdown a créé une structure intermédiaire)
        if not possible_paths:
             possible_paths = glob.glob(f"{base_path}/**/{self.seq_str}", recursive=True)

        # Filtrer pour ne garder que les vrais dossiers contenant 'velodyne'
        valid_paths = []
        for p in possible_paths:
            if os.path.exists(os.path.join(p, 'velodyne')):
                valid_paths.append(p)

        if not valid_paths:
            raise ValueError(f"Séquence {self.seq_str} introuvable dans {base_path}. Vérifiez que le dossier 'velodyne' est bien présent.")

        self.seq_path = valid_paths[0]
        print(f"Séquence chargée : {self.seq_path}")

        self.velo_path = os.path.join(self.seq_path, 'velodyne')
        self.label_path = os.path.join(self.seq_path, 'labels')
        self.poses_file = os.path.join(self.seq_path, 'poses.txt')
        self.calib_file = os.path.join(self.seq_path, 'calib.txt')

        # Fallback pour poses.txt/calib.txt s'ils sont dans le dossier parent (structure dataset/sequences/08)
        if not os.path.exists(self.poses_file):
             # Essayer de remonter d'un niveau (dataset/sequences/) ou deux
             parent = os.path.dirname(self.seq_path) # dataset/sequences
             grandparent = os.path.dirname(parent) # dataset

             # Cas dataset/poses.txt (peu probable mais...)
             # Cas dataset/sequences/08/poses.txt (standard)
             pass

        self.scan_files = sorted(glob.glob(os.path.join(self.velo_path, '*.bin')))
        self.label_files = sorted(glob.glob(os.path.join(self.label_path, '*.label')))
        self.poses = self._load_poses()
        self.calib = self._load_calib()

    def _load_poses(self):
        if not os.path.exists(self.poses_file):
            print(f"Info: poses.txt non trouvé ({self.poses_file}).")
            return []
        poses = []
        with open(self.poses_file, 'r') as f:
            for line in f:
                values = [float(v) for v in line.strip().split()]
                pose = np.vstack([np.array(values).reshape(3, 4), [0, 0, 0, 1]])
                poses.append(pose)
        return poses

    def _load_calib(self):
        if not os.path.exists(self.calib_file):
            print(f"Info: calib.txt non trouvé ({self.calib_file}).")
            return np.eye(4)
        calib = {}
        with open(self.calib_file, 'r') as f:
            for line in f:
                if ':' not in line: continue
                key, val = line.split(':', 1)
                calib[key] = np.array([float(x) for x in val.split()]).reshape(3, 4)
        if 'Tr' in calib:
            return np.vstack([calib['Tr'], [0, 0, 0, 1]])
        return np.eye(4)

    def get_scan(self, idx, apply_pose=True):
        scan = np.fromfile(self.scan_files[idx], dtype=np.float32).reshape(-1, 4)
        points = scan[:, :3]
        if apply_pose and self.poses and idx < len(self.poses):
            T = self.poses[idx] @ self.calib
            points = (T @ np.hstack([points, np.ones((len(points), 1))]).T).T[:, :3]
        return points

    def get_labels(self, idx):
        if idx >= len(self.label_files): return None, None
        label = np.fromfile(self.label_files[idx], dtype=np.uint32)
        return label & 0xFFFF, label >> 16

    def __len__(self): return len(self.scan_files)

In [ ]:
# @title 3.1 Construction du Nuage 4D
import numpy as np


# Mapping officiel SemanticKITTI (fusionne les classes "moving" avec leur équivalent statique)
LEARNING_MAP = {
  0: 0, 1: 0, 10: 1, 11: 2, 13: 5, 15: 3, 16: 5, 18: 4, 20: 5, 30: 6, 
  31: 7, 32: 8, 40: 9, 44: 10, 48: 11, 49: 12, 50: 13, 51: 14, 52: 0, 
  60: 9, 70: 15, 71: 16, 72: 17, 80: 18, 81: 19, 99: 0, 252: 1, 
  253: 7, 254: 6, 255: 8, 256: 5, 257: 5, 258: 4, 259: 5
}
# Vecteur de mapping rapide (taille max 260)
max_key = max(LEARNING_MAP.keys())
LABEL_MAP_ARRAY = np.zeros(max_key + 1, dtype=np.uint32)
for k, v in LEARNING_MAP.items():
    LABEL_MAP_ARRAY[k] = v

# SemanticKITTI classes "things" dans l'espace mappé (1 à 8)
# 1:car, 2:bicycle, 3:motorcycle, 4:truck, 5:other-vehicle, 6:person, 7:bicyclist, 8:motorcyclist
THINGS_CLASSES = set([1, 2, 3, 4, 5, 6, 7, 8])

# Initialisation des variables pour éviter les NameError
X_clustering = None
X_4d = None
Y_sem = None
Y_inst = None
Time_idx = None
Original_Indices = None # Pour garder la trace si on filtre

try:
    loader = SemanticKITTILoader("/content/semantic_kitti_data", SEQUENCE_TO_TEST)
    points_4d, gt_sem, gt_inst, times, indices = [], [], [], [], []
    
    total_points = 0
    actual_num_frames = len(loader) - START_FRAME if NUM_FRAMES == -1 else NUM_FRAMES
    print(f"Chargement frames {START_FRAME} -> {START_FRAME + actual_num_frames}...")
    for i in range(actual_num_frames):
        idx = START_FRAME + i
        if idx >= len(loader): break

        pts = loader.get_scan(idx, apply_pose=True)
        s_raw, inst = loader.get_labels(idx)
        
        # Mapping des labels sémantiques bruts vers l'espace d'évaluation
        s = LABEL_MAP_ARRAY[s_raw]

        # 4D Point: x, y, z, t
        t_col = np.full((len(pts), 1), i * DT_SCALE)
        pts_4d = np.hstack([pts, t_col])
        
        # Filtre Sémantique
        if SEMANTIC_MODE == "Oracle":
            # On ne garde que les classes "Things"
            mask = np.array([sem in THINGS_CLASSES for sem in s])
            pts_4d = pts_4d[mask]
            s = s[mask]
            inst = inst[mask]
            
            # Si on veut garder l'index original par rapport à la frame (pour de la visulaisation par ex)
            frame_indices = np.arange(len(mask))[mask]
        else:
            frame_indices = np.arange(len(pts))

        points_4d.append(pts_4d)
        gt_sem.append(s)
        gt_inst.append(inst)
        times.extend([i] * len(pts_4d))
        indices.append(frame_indices + total_points)
        total_points += len(pts) # On ajoute le total brut pour les indices absolus

    if points_4d:
        X_4d = np.vstack(points_4d)
        Y_sem = np.hstack(gt_sem)
        Y_inst = np.hstack(gt_inst)
        Time_idx = np.array(times)
        Original_Indices = np.hstack(indices)

        # Bird's Eye View : on utilise uniquement x, y, t pour le clustering
        if APPLY_BEV:
            print(f"Mode Bird's-Eye-View (BEV) activé : Clustering sur (x, y) [Streaming 2D BEV].")
            X_clustering = np.column_stack([X_4d[:, 0], X_4d[:, 1]])
        else:
            print("Mode 4D Complet activé : Clustering sur (x, y, z) [Streaming 3D].")
            X_clustering = X_4d

        print(f"Sémantique : Mode {SEMANTIC_MODE}.")
        print(f"Nuage 4D filtré: {X_4d.shape} points conservés.")
        print(f"Input Clustering: {X_clustering.shape}")
    else:
        print("Aucun point chargé ou aucun point 'thing' trouvé. Vérifiez les chemins.")

except Exception as e:
    print(f"Erreur lors du chargement des données: {e}")
    print("---------------------------------------------------------")
    print("⚠️ GÉNÉRATION DE DONNÉES SYNTHÉTIQUES (FALLBACK) ⚠️")
    print("---------------------------------------------------------")
    from sklearn.datasets import make_blobs
    n_samples = 5000
    X_syn, y_syn = make_blobs(n_samples=n_samples, n_features=3, centers=5, cluster_std=1.0)
    synth_frames = NUM_FRAMES if NUM_FRAMES != -1 else 10
    t_syn = np.random.randint(0, synth_frames, size=n_samples) * DT_SCALE
    X_4d = np.column_stack([X_syn, t_syn])
    Y_sem = np.zeros(n_samples, dtype=int)
    Y_inst = y_syn + 1 
    Time_idx = (t_syn / DT_SCALE).astype(int)
    X_clustering = X_4d[:, :3] if not APPLY_BEV else X_4d[:, [0, 1]]
    print(f"Données synthétiques générées: {X_clustering.shape}")

In [ ]:
# @title 4.1 HGP Clustering & Advanced 4D Panoptic Tracking (Kalman + OBB)
import time
import numpy as np
import os
from scipy.spatial import ConvexHull
from scipy.optimize import linear_sum_assignment
from shapely.geometry import Polygon

# --- Import sécurisé de HGPClusterer ---
try:
    from hgp_clusterer import HGPClusterer
except ImportError:
    print("⚠️ Module HGPClusterer introuvable. Tentative de correction du path...")
    import sys
    if "/content/HGP-clusterer/src" not in sys.path:
        sys.path.append("/content/HGP-clusterer/src")
    try:
        from hgp_clusterer import HGPClusterer
        print("✅ HGPClusterer importé avec succès après correction du path.")
    except ImportError as e:
        raise RuntimeError(f"❌ Impossible d'importer HGPClusterer même après correction. Erreur: {e}. Veuillez vérifier la compilation en section 1.5.")

# --- Paramètres de Clustering ---
K = 3 # @param {type:"integer"}
MIN_CLUSTER_SIZE = 1 # @param {type:"integer"}
EXP_Z = 1 # @param {type:"number"}
SPLITTING_MODE = "oracle_Gini" # @param ["None", "oracle_Gini"] {allow-input: true}
DBSCAN_FACTOR = 0.75 # @param {type:"number"}

CURRENT_GT_INSTANCES = None

def oracle_Gini(parent_pts_idx, children_pts_idx_list, ε_Gini=0.001):
    global CURRENT_GT_INSTANCES
    if CURRENT_GT_INSTANCES is None or len(parent_pts_idx) == 0: return False
    def get_gini(indices):
        if len(indices) == 0: return 0.0
        labels = CURRENT_GT_INSTANCES[indices]
        _, counts = np.unique(labels, return_counts=True)
        probs = counts / len(labels)
        return 1.0 - np.sum(probs**2)
    gini_p = get_gini(parent_pts_idx)
    if gini_p < ε_Gini: return False
    n_total = len(parent_pts_idx)
    gini_c_weighted = 0.0
    for c_pts in children_pts_idx_list:
        if len(c_pts) == 0: continue
        gini_c_weighted += (len(c_pts) / n_total) * get_gini(c_pts)
    return gini_c_weighted < (gini_p - ε_Gini)

SPLITTING_REGISTRY = {"None": None, "oracle_Gini": oracle_Gini}

# --- Priors de tailles et dynamiques par classe ---
# L, W, H : dimensions typiques
# max_speed : m/s (pour le gating adaptatif)
ALPINE_PRIORS = {
    1: {"name": "car",           "L": 4.5,  "W": 1.85, "H": 1.6,  "max_speed": 30.0},
    2: {"name": "bicycle",       "L": 1.8,  "W": 0.6,  "H": 1.1,  "max_speed": 10.0},
    3: {"name": "motorcycle",    "L": 2.2,  "W": 0.9,  "H": 1.3,  "max_speed": 25.0},
    4: {"name": "truck",         "L": 10.0, "W": 2.6,  "H": 3.5,  "max_speed": 25.0},
    5: {"name": "other-vehicle", "L": 12.0, "W": 2.6,  "H": 3.5,  "max_speed": 20.0},
    6: {"name": "person",        "L": 0.6,  "W": 0.6,  "H": 1.75, "max_speed": 2.5},
    7: {"name": "bicyclist",     "L": 1.8,  "W": 0.75, "H": 1.8,  "max_speed": 12.0},
    8: {"name": "motorcyclist",  "L": 2.2,  "W": 0.9,  "H": 1.8,  "max_speed": 25.0},
}

# --- Classes de Tracking ---

class KalmanTrack:
    """
    Track avec Filtre de Kalman Linéaire et Bounding Box 3D.
    État x = [x, y, z, vx, vy, vz, L, W, H]
    """
    def __init__(self, track_id, semantic_class, detection):
        self.track_id = track_id
        self.semantic_class = semantic_class
        
        # Initialisation de l'état (Position, Vitesse nulle, Dimensions initiales)
        c = detection["centroid"]
        dim = detection["dimensions"]
        self.x = np.array([c[0], c[1], c[2], 0, 0, 0, dim[0], dim[1], dim[2]], dtype=float)
        
        # Covariance initiale P
        self.P = np.eye(9) * 1.0
        self.P[3:6, 3:6] *= 10.0 # Grande incertitude sur la vitesse initiale
        
        # Matrices du modèle
        self.F = np.eye(9)  # Matrice de transition (mise à jour avec dt)
        self.H = np.zeros((6, 9)) # Matrice d'observation (on observe x, y, z, L, W, H)
        self.H[0:3, 0:3] = np.eye(3)
        self.H[3:6, 6:9] = np.eye(3)
        
        self.R = np.eye(6) * 0.1 # Bruit de mesure
        self.Q = np.eye(9) * 0.05 # Bruit de process
        
        self.age_of_occlusion = 0
        self.history = []

    def predict(self, dt):
        # Update transition matrix with dt
        self.F[0, 3] = dt
        self.F[1, 4] = dt
        self.F[2, 5] = dt
        
        # x = F * x
        self.x = self.F @ self.x
        # P = F * P * F.T + Q
        self.P = self.F @ self.P @ self.F.T + self.Q
        self.age_of_occlusion += 1

    def update(self, detection, dt):
        # Mesure z = [x, y, z, L, W, H]
        c = detection["centroid"]
        dim = detection["dimensions"]
        z = np.array([c[0], c[1], c[2], dim[0], dim[1], dim[2]])
        
        # Gain de Kalman K = P * H.T * inv(H * P * H.T + R)
        S = self.H @ self.P @ self.H.T + self.R
        K = self.P @ self.H.T @ np.linalg.inv(S)
        
        # Innovation y = z - H * x
        y = z - self.H @ self.x
        
        # x = x + K * y
        self.x = self.x + K @ y
        # P = (I - K * H) * P
        I = np.eye(9)
        self.P = (I - K @ self.H) @ self.P
        
        self.age_of_occlusion = 0

    def get_centroid(self): return self.x[0:3]
    def get_dimensions(self): return self.x[6:9]
    def get_velocity(self): return self.x[3:6]

def estimate_dimensions(pts_3d):
    """Calcule les dimensions (L, W, H) alignées sur les axes (simplification OBB)"""
    if len(pts_3d) == 0: return np.array([1.0, 1.0, 1.0])
    mins = np.min(pts_3d, axis=0)
    maxs = np.max(pts_3d, axis=0)
    return max(0.1, maxs[0] - mins[0]), max(0.1, maxs[1] - mins[1]), max(0.1, maxs[2] - mins[2])

def compute_mahalanobis_distance(track, detection):
    """Calcule la distance de Mahalanobis entre un track et une détection"""
    c = detection["centroid"]
    dim = detection["dimensions"]
    z = np.array([c[0], c[1], c[2], dim[0], dim[1], dim[2]])
    
    # Projection de l'état dans l'espace de mesure
    z_pred = track.H @ track.x
    S = track.H @ track.P @ track.H.T + track.R
    
    delta = z - z_pred
    try:
        dist = np.sqrt(delta.T @ np.linalg.inv(S) @ delta)
    except:
        dist = np.inf
    return dist

# --- Boucle Principale de Tracking ---

if X_clustering is None or len(X_clustering) == 0:
    raise RuntimeError("X_clustering est vide.")

labels_pred = np.full(len(X_clustering), -1, dtype=int)
raw_labels = np.full(len(X_clustering), -1, dtype=int)
next_raw_id = 1
next_track_id = 1
active_tracks = []
MAX_AGE = 5
DT = 0.1 # 10 Hz

frames = np.unique(Time_idx)
print(f"Frames à traiter : {frames}")

for t in frames:
    print(f"\r=== Frame {t} | Tracks: {len(active_tracks)} ===", end="")
    frame_mask = (Time_idx == t)
    X_frame = X_clustering[frame_mask]
    Y_sem_frame = Y_sem[frame_mask]
    Y_inst_frame = Y_inst[frame_mask]
    global_indices = np.where(frame_mask)[0]
    
    # 1. Prédiction Kalman
    for track in active_tracks:
        track.predict(DT)

    unique_classes = np.unique(Y_sem_frame)
    for semantic_class in unique_classes:
        if semantic_class not in THINGS_CLASSES and SEMANTIC_MODE == "Oracle":
            continue
            
        class_mask = (Y_sem_frame == semantic_class)
        X_class = X_frame[class_mask]
        Y_inst_class = Y_inst_frame[class_mask]
        class_global_indices = global_indices[class_mask]
        CURRENT_GT_INSTANCES = Y_inst_class
        
        if len(X_class) < MIN_CLUSTER_SIZE: continue
            
        # 2. HGP Clustering
        prior = ALPINE_PRIORS.get(semantic_class, {"W": 1.0, "max_speed": 20.0})
        clusterer = HGPClusterer(
            K=K, min_cluster_size=MIN_CLUSTER_SIZE, method=prior["W"] * DBSCAN_FACTOR,
            splitting=SPLITTING_REGISTRY.get(SPLITTING_MODE),
            backend=BACKEND, verbose=False
        )
        try:
            class_labels_pred = clusterer.fit_predict(X_class)
        except: continue

        valid_cluster_mask = class_labels_pred >= 0
        unique_clusters_class = np.unique(class_labels_pred[valid_cluster_mask])
        
        # 3. Préparation des Détections (Centroid + Dimensions OBB)
        detections = []
        for cid in unique_clusters_class:
            m_local = (class_labels_pred == cid)
            raw_labels[class_global_indices[m_local]] = next_raw_id
            pts_3d = X_4d[class_global_indices][m_local][:, :3]
            
            detections.append({
                "centroid": np.median(pts_3d, axis=0),
                "dimensions": estimate_dimensions(pts_3d),
                "mask_local": m_local
            })
            next_raw_id += 1
            
        class_tracks = [tr for tr in active_tracks if tr.semantic_class == semantic_class]
        n_tracks, n_dets = len(class_tracks), len(detections)
        
        matched_tr_idx, matched_det_idx = set(), set()
        
        if n_tracks > 0 and n_dets > 0:
            # 4. Association par Distance de Mahalanobis + Gating Adaptatif
            cost_matrix = np.full((n_tracks, n_dets), 1e6)
            gate_dist = prior["max_speed"] * DT * 1.5 # Gate de sécurité
            
            for i, tr in enumerate(class_tracks):
                for j, det in enumerate(detections):
                    # Gating Euclidien simple (pré-filtre)
                    dist_euc = np.linalg.norm(tr.get_centroid()[:2] - det["centroid"][:2])
                    if dist_euc > gate_dist: continue
                    
                    # Distance de Mahalanobis (KF)
                    cost_matrix[i, j] = compute_mahalanobis_distance(tr, det)
            
            row_ind, col_ind = linear_sum_assignment(cost_matrix)
            for r, c in zip(row_ind, col_ind):
                if cost_matrix[r, c] < 15.0: # Seuil Mahalanobis (typique pour 6 DoF)
                    tr, det = class_tracks[r], detections[c]
                    tr.update(det, DT)
                    labels_pred[class_global_indices[det["mask_local"]]] = tr.track_id
                    matched_tr_idx.add(r)
                    matched_det_indices.add(c) # Note: matched_det_indices n'est pas utilisé ici, correction:
                    matched_det_idx.add(c)

        # 5. Gestion des Naissances
        for j, det in enumerate(detections):
            if j not in matched_det_idx:
                new_tr = KalmanTrack(next_track_id, semantic_class, det)
                active_tracks.append(new_tr)
                labels_pred[class_global_indices[det["mask_local"]]] = next_track_id
                next_track_id += 1

    # 6. Nettoyage des pistes mortes
    active_tracks = [tr for tr in active_tracks if tr.age_of_occlusion <= MAX_AGE]

print(f"\nTracking terminé. Total IDs : {next_track_id - 1}")



In [ ]:
# @title 5.1 Évaluation Officielle SemanticKITTI (PQ, SQ, RQ)
import os
import shutil
import numpy as np
import yaml

if X_clustering is not None and len(X_clustering) > 0:
    print("Préparation des fichiers pour l'évaluation officielle (semantic-kitti-api)...")
    
    eval_dir = "/content/eval_data"
    pred_dir = "/content/eval_predictions"
    seq_str = f"{SEQUENCE_TO_TEST:02d}"
    
    gt_labels_dir = os.path.join(eval_dir, "sequences", seq_str, "labels")
    pred_labels_dir = os.path.join(pred_dir, "sequences", seq_str, "predictions")
    
    # Nettoyage précédent éventuel
    shutil.rmtree(eval_dir, ignore_errors=True)
    shutil.rmtree(pred_dir, ignore_errors=True)
    
    os.makedirs(gt_labels_dir, exist_ok=True)
    os.makedirs(pred_labels_dir, exist_ok=True)
    
    # Création d'une configuration personnalisée pour n'évaluer que cette séquence
    custom_cfg_path = "/content/custom_eval_config.yaml"
    with open("/content/semantic-kitti-api/config/semantic-kitti.yaml", 'r') as f:
        cfg = yaml.safe_load(f)
    cfg['split']['valid'] = [SEQUENCE_TO_TEST]
    with open(custom_cfg_path, 'w') as f:
        yaml.dump(cfg, f)
    
    # On reconstruit les labels prédits frame par frame
    actual_num_frames = len(loader) - START_FRAME if NUM_FRAMES == -1 else NUM_FRAMES
    for i in range(actual_num_frames):
        idx = START_FRAME + i
        if idx >= len(loader): break
        
        # Copie du GT
        gt_file = loader.label_files[idx]
        shutil.copy(gt_file, os.path.join(gt_labels_dir, os.path.basename(gt_file)))
        
        # Récupération des labels originaux pour garder la sémantique de fond
        s_raw, _ = loader.get_labels(idx)
        s_mapped = LABEL_MAP_ARRAY[s_raw]
        
        if SEMANTIC_MODE == "Oracle":
            mask = np.array([sem in THINGS_CLASSES for sem in s_mapped])
            frame_indices = np.where(mask)[0]
        else:
            frame_indices = np.arange(len(s_raw))
            
        # Initialisation de la prédiction
        pred_label = s_raw.astype(np.uint32)
        
        mask_time = (Time_idx == i)
        inst_preds = labels_pred[mask_time]
        
        valid_inst_mask = inst_preds >= 0
        valid_local_indices = frame_indices[valid_inst_mask]
        # Offset +1 car l'ID 0 est réservé au background
        valid_inst_ids = inst_preds[valid_inst_mask] + 1 
        
        pred_label[valid_local_indices] = (s_raw[valid_local_indices] & 0xFFFF) | (valid_inst_ids.astype(np.uint32) << 16)
        
        # Sauvegarde
        pred_filename = os.path.join(pred_labels_dir, os.path.basename(gt_file))
        pred_label.tofile(pred_filename)
        
    print("Fichiers de prédiction générés. Lancement de evaluate_panoptic.py...")
    os.makedirs("/content/eval_output", exist_ok=True)
    # On lance le script officiel sur ce mini-dataset de test
    !python /content/semantic-kitti-api/evaluate_panoptic.py --dataset {eval_dir} --predictions {pred_dir} --split valid --data_cfg {custom_cfg_path} --output /content/eval_output
    if os.path.exists("/content/eval_output/scores.txt"):
        print("\n--- RÉSULTATS DE L'ÉVALUATION ---")
        with open("/content/eval_output/scores.txt", "r") as f:
            print(f.read())
else:
    print("Pas de données pour l'évaluation.")


In [ ]:
# @title 6.1 Visualisation 2D (Vue du Dessus)
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

if X_4d is not None and len(X_4d) > 0:
    max_points = 100000
    num_points = len(X_4d)
    if num_points > max_points:
        print(f"Sous-échantillonnage de {num_points} à {max_points} points...")
        idx = np.random.choice(num_points, max_points, replace=False)
    else:
        idx = np.arange(num_points)
    
    X_v = X_4d[idx]
    L_v = labels_pred[idx]
    GT_v = Y_inst[idx]
    S_v = Y_sem[idx]
    R_v = raw_labels[idx]
    
    THINGS_NAMES = {
        1: "car", 2: "bicycle", 3: "motorcycle", 4: "truck",
        5: "other-vehicle", 6: "person", 7: "bicyclist", 8: "motorcyclist"
    }
    
    fig = make_subplots(
        rows=1, cols=3,
        subplot_titles=('Ground Truth', 'Raw Clustering', 'Panoptic Tracking Result'),
        horizontal_spacing=0.05
    )
    
    # --- Gauche : Ground Truth ---
    u_gt = np.unique(GT_v)
    for l in u_gt:
        if l <= 0: continue
        m = GT_v == l
        class_name = THINGS_NAMES.get(S_v[m][0], "unknown")
        hover_text = [f"Class: {class_name}<br>GT Instance: {l}<br>x: {X_v[m][i, 0]:.2f}<br>y: {X_v[m][i, 1]:.2f}" for i in range(np.sum(m))]
        fig.add_trace(go.Scatter(
            x=X_v[m, 0], y=X_v[m, 1],
            mode='markers', marker=dict(size=3), name=f'GT {l}',
            text=hover_text, hoverinfo='text+name'
        ), row=1, col=1)
    
    # --- Milieu : Raw Clustering ---
    u_r = np.unique(R_v)
    for l in u_r:
        if l == -1: continue
        m = R_v == l
        class_name = THINGS_NAMES.get(S_v[m][0], "unknown")
        hover_text = [f"Class: {class_name}<br>Raw ID: {l}<br>x: {X_v[m][i, 0]:.2f}<br>y: {X_v[m][i, 1]:.2f}" for i in range(np.sum(m))]
        fig.add_trace(go.Scatter(
            x=X_v[m, 0], y=X_v[m, 1],
            mode='markers', marker=dict(size=3), name=f'Raw {l}',
            text=hover_text, hoverinfo='text+name'
        ), row=1, col=2)
    
    # --- Droite : Tracking Result ---
    u_l = np.unique(L_v)
    for l in u_l:
        if l == -1: continue
        m = L_v == l
        class_name = THINGS_NAMES.get(S_v[m][0], "unknown")
        hover_text = [f"Class: {class_name}<br>Track ID: {l}<br>x: {X_v[m][i, 0]:.2f}<br>y: {X_v[m][i, 1]:.2f}" for i in range(np.sum(m))]
        fig.add_trace(go.Scatter(
            x=X_v[m, 0], y=X_v[m, 1],
            mode='markers', marker=dict(size=3), name=f'Track {l}',
            text=hover_text, hoverinfo='text+name'
        ), row=1, col=3)
    
    # Configuration orthonormale et mise en page
    fig.update_xaxes(scaleanchor="y", scaleratio=1, title_text="X (m)")
    fig.update_yaxes(title_text="Y (m)")
    
    fig.update_layout(
        title_text="Comparaison Bird's Eye View (BEV) - Séquence SemanticKITTI",
        height=700,
        showlegend=False,
        template="plotly_dark"
    )
    fig.show()
else:
    print("Pas de données à afficher.")

